In [4]:
# 创建Agent

## 定义工具函数
## 使用 `@tool` 装饰器，把普通Python函数变成Agent可以调用的工具


from dotenv import load_dotenv
from langchain.tools import tool

load_dotenv()


True

In [6]:
# 步骤1：用 `@tool` 装饰器定义一个工具
# 函数的文档字符串就是工具的描述
# 模型会根据描述来判断何时调用这个工具

@tool
def get_weather(city: str) -> str:
    '''
    查询指定城市的天气情况
    Args:
        city: 城市名称，如 杭州、北京
    '''
    # 模拟数据演示
    weather_data = {
        "杭州": "晴，25℃，湿度60%",
        "北京": "多云，18℃，湿度45%",
        "上海": "小雨，22℃，湿度80%",
    }

    return weather_data.get(city, f"未找到{city}的天气数据")

@tool
def calculate(expression: str) -> str:
    '''
    执行数学计算。支持加减乘除等基本运算
    Args:
        expression: 数学表达式，如 "3 * 7 + 2
    '''
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果：{expression} = {result}"
    except Exception as e:
        return f"计算错误：{e}"

**文档字符串（函数的""" ... """部分）非常重要。模型会读取工具的描述来决定是否调用这个工具以及传什么参数。描述越清晰，模型就越不容易出错。**

In [7]:
# 步骤2：创建Agent
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 初始化模型
model = init_chat_model("deepseek-v4-flash")

# 创建Agent，传入模型和工具列表
agent = create_agent(
    model=model,
    tools=[get_weather, calculate],     # 工具列表，Agent 可以调用这些工具
    system_prompt="你是一个乐于助人的助手，会使用工具来回答问题。"     # 系统提示词，定义 Agent 的角色和行为
)

In [8]:
# 步骤3：运行Agent
# 构建输入消息
# 消息列表中的第一条通常式 HumanMessage（用户消息）
from langchain.messages import HumanMessage

inputs = {"messages": [HumanMessage(content="杭州天气怎么样？")]}

# invoke() 运行Agent，返回最终状态
result = agent.invoke(inputs)

# 查看消息历史（包含AI的工具调用和工具返回结果）
print("=== 完整消息历史 ===")
for msg in result["messages"]:
    print(f"[{msg.type}] {msg.content[:100]}")  # 截取前100字符

print("\n=== 最终回复 ===")
# 最后一条AI消息就是最终答案
print(result["messages"][-1].content)

=== 完整消息历史 ===
[human] 杭州天气怎么样？
[ai] 好的，我马上帮你查询杭州的天气情况！
[tool] 晴，25℃，湿度60%
[ai] 杭州目前的天气情况如下：

🌤 **天气：** 晴  
🌡 **温度：** 25℃  
💧 **湿度：** 60%

天气晴朗，温度舒适，很适合外出活动哦！不过湿度略高，体感可能会稍有点闷。请问还有什

=== 最终回复 ===
杭州目前的天气情况如下：

🌤 **天气：** 晴  
🌡 **温度：** 25℃  
💧 **湿度：** 60%

天气晴朗，温度舒适，很适合外出活动哦！不过湿度略高，体感可能会稍有点闷。请问还有什么可以帮你的吗？😊


In [9]:
# Agent调用多个工具
inputs = {"messages": [HumanMessage(
    content="杭州和北京今天温差多少度？"
)]}
result = agent.invoke(inputs)

print("=== 完整消息历史 ===")
for msg in result["messages"]:
    if msg.type == "tool":
        print(f"[tool {msg.name}] {msg.content}")
    else:
        print(f"[{msg.type}] {msg.content[:120]}")

print("\n=== 最终回复 ===")
print(result["messages"][-1].content)

=== 完整消息历史 ===
[human] 杭州和北京今天温差多少度？
[ai] 好的，我来查询杭州和北京今天的天气情况，然后计算温差。
[tool get_weather] 晴，25℃，湿度60%
[tool get_weather] 多云，18℃，湿度45%
[ai] 好的，我来计算一下温差：
[tool calculate] 计算结果：25 - 18 = 7
[ai] 好的，查询结果如下：

### 🌡️ 杭州 vs 北京 今日温差

| 城市 | 天气 | 温度 |
|:---:|:---:|:---:|
| 🏙️ **杭州** | ☀️ 晴 | **25℃** |
| 🏙️ **北京** | ⛅ 多云

=== 最终回复 ===
好的，查询结果如下：

### 🌡️ 杭州 vs 北京 今日温差

| 城市 | 天气 | 温度 |
|:---:|:---:|:---:|
| 🏙️ **杭州** | ☀️ 晴 | **25℃** |
| 🏙️ **北京** | ⛅ 多云 | **18℃** |

> **温差：7℃**（杭州比北京暖7度）

今天杭州天气晴朗、气温舒适，北京则多云、稍凉一些。如果您要往返两地，记得适当调整着装哦！😊


In [11]:
# 完整代码
from dotenv import load_dotenv
load_dotenv()

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage


# 定义工具
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气情况。

    Args:
        city: 城市名称，如 "杭州"、"北京"
    """
    weather_data = {
        "杭州": "晴，25°C，湿度 60%",
        "北京": "多云，18°C，湿度 45%",
        "上海": "小雨，22°C，湿度 80%",
    }
    return weather_data.get(city, f"未找到 {city} 的天气数据")


@tool
def calculate(expression: str) -> str:
    """执行数学计算。支持加减乘除等基本运算。

    Args:
        expression: 数学表达式，如 "3 * 7 + 2"
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"


# 创建 Agent
model = init_chat_model("deepseek-v4-flash")
agent = create_agent(
    model=model,
    tools=[get_weather, calculate],
    system_prompt="你是一个乐于助人的助手，会使用工具来回答问题。",
)

# 运行 Agent
def ask(question: str):
    """发送问题到 Agent 并打印结果"""
    inputs = {"messages": [HumanMessage(content=question)]}
    result = agent.invoke(inputs)
    print(f"问题: {question}")
    print(f"回答: {result['messages'][-1].content}")
    print("-" * 50)
    return result


# 测试几个问题
ask("杭州今天天气怎么样？")
ask("杭州和北京今天温差多少度？")
# 第三个问题：Agent 能够理解"推荐人数"是一个计算问题，并自动调用 calculate 工具。这说明工具调用的决策是由模型对语义的理解来驱动的，而不是硬编码的规则。
ask("菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？")

问题: 杭州今天天气怎么样？
回答: 杭州今天天气不错！☀️

- **天气状况**：晴
- **温度**：25°C
- **湿度**：60%

是个阳光明媚的好天气，温度也很舒适，适合外出活动哦！不过出门记得做好防晒~😊
--------------------------------------------------
问题: 杭州和北京今天温差多少度？
回答: 今天杭州和北京的温差是 **7°C**。

- **杭州**：🌤 晴，25°C
- **北京**：⛅ 多云，18°C
- **温差**：25°C - 18°C = **7°C**

杭州比北京暖和不少，如果从北京去杭州记得少穿点哦～
--------------------------------------------------
问题: 菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？
回答: 没错！计算结果也是 **5**。所以一共有 **5 个人**推荐菜鸟教程 RUNOOB 这个学习平台！🎉

菜鸟教程确实是个很棒的宝藏网站，适合新手入门编程，有 HTML、CSS、Python、Java 等各种教程，感谢你和你朋友们的推荐～ 😄
--------------------------------------------------


{'messages': [HumanMessage(content='菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？', additional_kwargs={}, response_metadata={}, id='befaf955-e188-4d03-8a6e-526823fdf168'),
  AIMessage(content='这是一个简单的数学问题！3 个朋友推荐 + 另外 2 个人推荐 = **5 个人推荐** 👍\n\n让我用计算工具验证一下：', additional_kwargs={'refusal': None, 'reasoning_content': '用户问了一个问题：菜鸟教程 RUNOOB 是一个很棒的学习平台，如果我有3个朋友都推荐了，再加上2个，一共多少人推荐？\n\n这其实是一个简单的数学问题：3 + 2 = 5。\n\n我可以直接回答，不需要调用工具。但用户的问题包含了一个计算，我可以使用计算工具来验证一下。\n\n让我计算一下 3 + 2。'}, response_metadata={'token_usage': {'completion_tokens': 160, 'prompt_tokens': 406, 'total_tokens': 566, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 81, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 150}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_2026

In [13]:
# Agent 支持异步模式，适合在 Web 服务等异步环境中使用
import asyncio
from langchain.messages import HumanMessage


async def main():
    # ainvoke() 是 invoke() 的异步版本
    inputs = {"messages": [HumanMessage(content="杭州天气怎么样？")]}
    result = await agent.ainvoke(inputs)
    print(result["messages"][-1].content)


# 运行异步函数
# asyncio.run(main())
# 直接 await（在Jupyter 中）
result = await main()


杭州今天的天气是**晴天**，气温约为 **25°C**，湿度 **60%**，天气不错，适合出行！☀️


In [ ]:
# LangChain 模型调用 -- init_chat_model() 函数
## 用统一的方式连接 20 多种模型提供商，不需要记忆每个提供商的类名和参数差异。

## init_chat_model() 函数有两种使用模式：

In [ ]:
### 1：固定模型
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

# 指定model模型，返回固定模型
model = init_chat_model('deepseek:deepseek-v4-flash', temperature=0.7)
resp = model.invoke('介绍菜鸟教程 RUNOOB')
print(resp.content)

In [ ]:
### 2：可配置模型
### 不指定 model（或设为 None），创建可在运行时动态切换的模型：
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
# 不指定 model，返回可配置模型
# 可以固定一些参数（如 temperature=0.7），其余运行时指定
configurable_model = init_chat_model(temperature=0.7)

# 运行时通过 config 指定模型
response = configurable_model.invoke(
    "介绍菜鸟教程 RUNOOB",
    config={"configurable": {"model": "deepseek-v4-flash"}}
)
print(response.content)

# 同一个模型实例，可以用不同的模型来执行
response = configurable_model.invoke(
    "介绍菜鸟教程 RUNOOB",
    config={"configurable": {"model": "claude-sonnet-4-5"}}
)
print(response.content)

常用kwargs参数
kwargs 参数会直接传递给底层模型类，常用的包括：

```{table}
| 参数 |	类型 | 说明 | 适用提供商 |
| temperature |	float |	控制随机性，0~2，默认值因模型而异 | 大部分 |
| max_tokens | int | 限制输出最大 Token 数 | 全部 |
| timeout | int 或 float | 请求超时秒数 | 全部 |
| max_retries | int | 请求失败后的重试次数 | 大部分 |
| base_url | str | 自定义 API 端点 | 大部分 |
| rate_limiter | BaseRateLimiter | 速率限制器实例 | 大部分 |
| top_p | float | 核采样参数，0~1 | 大部分 |
| stop | list[str] | 停止序列，模型遇到这些词时停止生成 | 大部分|
```

ConfigurableModel——运行时切换模型


In [ ]:
from langchain.chat_models import init_chat_model

# 创建可配置模型，并设置默认值
model = init_chat_model(
    "deepseek:deepseek-v4-flash",       # 默认模型
    configurable_fields="any",  # 所有参数都可在运行时修改
    config_prefix="my",         # 配置键前缀
    temperature=0.3,            # 默认温度
)

# 使用默认配置运行
response = model.invoke("介绍菜鸟教程")
print(f"默认配置: {response.content[:50]}...")

# 运行时覆盖模型和参数（注意 my_ 前缀）
response = model.invoke(
    "介绍菜鸟教程 RUNOOB",
    config={
        "configurable": {
            "my_model": "deepseek:deepseek-v4-pro",       # 切换模型
            "my_temperature": 0.9,             # 调整温度
        }
    }
)
print(f"覆盖配置: {response.content[:50]}...")